In [12]:
import open3d as o3d

dataset_name = "stereo_tambor"
DATASETS_PATH = "datasets"

pcd_path = f"{DATASETS_PATH}/{dataset_name}/data/results/tambor-pcd-cleaned.ply"

pcd = o3d.io.read_point_cloud(pcd_path)
print(pcd)

PointCloud with 423288 points.


In [13]:
down_pcd = pcd.voxel_down_sample(voxel_size=2)
print(down_pcd)

PointCloud with 54585 points.


In [14]:
import numpy as np

down_pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=10.0, max_nn=30)
)

In [15]:
down_pcd.orient_normals_consistent_tangent_plane(100)

In [16]:
depth = 9
with o3d.utility.VerbosityContextManager(o3d.utility.VerbosityLevel.Debug) as cm:
    tambor_mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        down_pcd, depth=depth, linear_fit=True
    )

[Open3D DEBUG] Input Points / Samples: 54585 / 54583
[Open3D DEBUG] #   Got kernel density: 0.1312260627746582 (s), 426.94921875 (MB) / 426.94921875 (MB) / 587 (MB)
[Open3D DEBUG] #     Got normal field: 0.1345818042755127 (s), 436.94921875 (MB) / 436.94921875 (MB) / 587 (MB)
[Open3D DEBUG] Point weight / Estimated Area: 1.016833e-04 / 5.550385e+00
[Open3D DEBUG] #       Finalized tree: 0.24982190132141113 (s), 438.82421875 (MB) / 438.82421875 (MB) / 587 (MB)
[Open3D DEBUG] #  Set FEM constraints: 0.13929295539855957 (s), 438.9765625 (MB) / 438.9765625 (MB) / 587 (MB)
[Open3D DEBUG] #Set point constraints: 0.048458099365234375 (s), 438.9765625 (MB) / 438.9765625 (MB) / 587 (MB)
[Open3D DEBUG] Leaf Nodes / Active Nodes / Ghost Nodes: 1064043 / 372592 / 843457
[Open3D DEBUG] Memory Usage: 438.977 MB
Cycle[0] Depth[0/9]:	Updated constraints / Got system / Solved in:  0.000 /  0.000 /  0.000	(438.977 MB)	Nodes: 8
CG: 2.0296e+00 -> 2.0296e+00 -> 1.9632e-03 (9.7e-04) [0]
Cycle[0] Depth[1/9]:

In [17]:
vertices_to_remove = densities < np.quantile(densities, 0.01)
tambor_mesh.remove_vertices_by_mask(vertices_to_remove)
tambor_mesh.paint_uniform_color([0.8, 0.8, 0.8])

mesh_path = f"{DATASETS_PATH}/{dataset_name}/data/results/tambor_mesh.ply"

o3d.io.write_triangle_mesh(mesh_path, tambor_mesh)

True